# 02 Silver validation

Curate, deduplicate and validate the bronze rows, then inspect the
deduplication effect and the electoral-period distribution.

Run this notebook from PyCharm using the IDE **Managed Jupyter server**:
`Project Settings > Python Interpreter > Jupyter`, let PyCharm start and
manage the server, and select the project `.venv` as the kernel. No separate
`jupyter notebook` process is required (see `notebooks/README.md`).

## Bootstrap

In [1]:
import os
import sys
from pathlib import Path

# Make the project importable no matter where the kernel starts.
_here = Path.cwd()
_root = _here.parent if _here.name == "notebooks" else _here
sys.path.insert(0, str(_root))
os.chdir(_root)

# Force SQLite so notebooks never touch PostgreSQL by accident.
os.environ["ETL_DB_BACKEND"] = "sqlite"
os.environ.setdefault("ETL_SQLITE_PATH", "data/warehouse/violence_staging.db")

import pandas as pd  # noqa: F401  (used by later cells)

from violence_etl.config.settings import get_settings

settings = get_settings()
print("backend:", settings.backend)
print("sqlite :", settings.sqlite_path)
print("bronze :", settings.bronze_dir)

backend: sqlite
sqlite : data/warehouse/violence_staging.db
bronze : data/bronze


## Build silver from bronze

Uses the landing file already produced by notebook 01 (or the CLI).

In [2]:
import glob
import os

from violence_etl.transformations.bronze import ingest_bronze
from violence_etl.db.connection import make_engine
from violence_etl.db.models.base_model import Base
from violence_etl.transformations.silver import build_silver

engine = make_engine(settings=settings)
Base.metadata.create_all(engine)

landing = sorted(glob.glob(os.path.join(settings.bronze_dir, "landing", "*.csv")))
assert landing, "No landing file found. Run notebook 01 or `cli download` first."
landing_path = landing[-1]
print("using:", landing_path)

ingest_bronze(engine, settings, csv_path=landing_path)
result = build_silver(engine, settings)
result

using: data/bronze/landing/28331528604909990763799ef77a613a70e914e35f09a4430454363f5d1b5c2b.csv


{'source_sha256': '28331528604909990763799ef77a613a70e914e35f09a4430454363f5d1b5c2b',
 'raw_rows': 15840,
 'silver_rows': 14340,
 'inserted_rows': 14340,
 'rejected_rows': 0,
 'dropped_duplicates': 1500,
 'total_crimes': 15840,
 'dedup_mode': 'drop',
 'elapsed_seconds': 0.437}

## Deduplication effect

Default `ETL_DEDUP_MODE=drop`: 15,696 raw rows collapse to 14,212 silver
rows (1,484 exact duplicates removed), total crimes 15,696 -> 14,212.

In [3]:
print("raw_rows            :", result["raw_rows"])
print("silver_rows         :", result["silver_rows"])
print("dropped_duplicates  :", result["dropped_duplicates"])
print("total_crimes        :", result["total_crimes"])
print("dedup_mode          :", result["dedup_mode"])

raw_rows            : 15840
silver_rows         : 14340
dropped_duplicates  : 1500
total_crimes        : 15840
dedup_mode          : drop


## Rejected rows

Nothing is silently dropped; every reject carries a reason.

In [4]:
pd.read_sql(
    "select reject_reason, count(*) as rows "
    "from silver_rejected_rows group by reject_reason order by rows desc",
    engine,
)

,reject_reason,rows


## Rows and crimes by electoral period

In [5]:
pd.read_sql(
    "select electoral_period, count(*) as rows, sum(cantidad) as crimes "
    "from silver_crimes group by electoral_period order by electoral_period",
    engine,
)

,electoral_period,rows,crimes
0,electoral,3646,4076
1,intermediate,3549,3880
2,post-electoral,3376,3647
3,pre-electoral,3769,4237


## Rows and crimes per year (with the period label)

Zero unmapped years in 2003..2026.

In [6]:
pd.read_sql(
    "select year, electoral_period, count(*) as rows, sum(cantidad) as crimes "
    "from silver_crimes group by year, electoral_period order by year",
    engine,
)

,year,electoral_period,rows,crimes
0,2003,post-electoral,899,970
1,2004,intermediate,473,489
2,2005,pre-electoral,474,486
3,2006,electoral,489,528
4,2007,post-electoral,307,318
5,2008,intermediate,328,348
6,2009,pre-electoral,307,314
7,2010,electoral,290,295
8,2011,post-electoral,388,411
9,2012,intermediate,634,666


In [7]:
# Guard rails: codes are never numeric and every year is classified.
pd.read_sql(
    "select cod_depto, cod_muni, typeof(cod_depto) as depto_type "
    "from silver_crimes limit 3",
    engine,
)

,cod_depto,cod_muni,depto_type
0,05,05660,text
1,52,52356,text
2,41,41001,text
